# Practice Workshop 5: Merging and combining data

These exercises go one step further than the weekly practice workshops and are
closer to the course workshops in length and difficulty. Each exercise uses one
of the data sets in the course's `data/` folder; the setup cell defines
`DATA_PATH` so that file paths work from this folder.

Before running each result, predict what you expect to see.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = '../../data'

***
# Exercise 1: Output gap and unemployment gap (Okun's law)

*Potential GDP* (`GDPPOT`) is the US Congressional Budget Office's estimate of
what the economy would produce at normal capacity. The *output gap* measures how
far actual GDP is from this level. *Okun's law* says that when output is below
potential, unemployment is above its natural rate.

You need four files from `FRED/`: `GDP.csv` and `GDPPOT.csv` (quarterly),
`NROU.csv` (quarterly natural rate of unemployment) and `UNRATE.csv` (monthly).

1.  Load all four files and parse `DATE` as dates. For each, report the number of
    rows and the first and last date.

2.  `UNRATE` is monthly, while the other series are quarterly. Convert it to
    quarterly averages using [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) with the
    rule `'QS'` (quarter start), so that the dates match the quarterly files.

    *Hint:* `resample()` needs dates as the index.

3.  Merge `GDP` and `GDPPOT` on `DATE` using an inner join and an outer join.
    Explain the difference in the number of rows. What do the extra rows in the
    outer join contain, and why do they exist?

    *Hint:* Look at the last dates of `GDPPOT`. The CBO also publishes estimates
    of potential output for future quarters.

4.  Merge all four series into one DataFrame with an inner join. Which period
    does the result cover?

5.  Compute the output gap in percent, `100 * (GDP - GDPPOT) / GDPPOT`, and the
    unemployment gap, `UNRATE - NROU`. Which quarter had the most negative output
    gap?

6.  Compute the correlation between the two gaps and create a scatter plot with
    the output gap on the horizontal axis and the unemployment gap on the vertical
    axis. Add horizontal and vertical lines at zero. Is Okun's law supported?

7.  **[Advanced]** Use [`np.polyfit(x, y, 1)`](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html) to fit a
    straight line through the scatter plot and add it to the figure. By how many
    percentage points does unemployment rise above its natural rate when output
    falls 1% below potential?

***
# Exercise 2: Annual stock returns

You compare the annual returns of five tech stocks with the S&P 500 index. The
files are in `stockmarket/`: `AAPL.csv`, `MSFT.csv`, `NVDA.csv`, `TSLA.csv`,
`META.csv` (columns `Date`, `Open`, `Close`) and `SP500.csv` (columns `Date`,
`SP500`).

1.  Write a loop over the five ticker symbols. For each one:

    1.  load the file with `Date` parsed as dates and set as the index,
    2.  get the last closing price of each calendar year using
        [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) with the rule `'YE'` and
        `last()`,
    3.  compute the annual return in percent with `pct_change()`, and
    4.  rename the resulting Series to the ticker symbol and append it to a list.

2.  Combine the five Series into one DataFrame with
    [`concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) along the columns (`axis=1`). Do this twice: once with the
    default outer join and once with `join='inner'`. How many years does each
    result contain? Why does the outer join contain missing values?

3.  Drop all years with a missing return for at least one stock. For each
    remaining year, which stock had the highest return? Count how many years each
    stock was the best.

    *Hint:* `idxmax(axis=1)` returns the column label of the largest value in
    each row.

4.  Compute the annual return of the S&P 500 in the same way and merge it with
    your stock returns using the dates as keys.

    *Hint:* Use [`merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) with `left_index=True` and `right_index=True`, or
    [`join()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html).

    In how many years did each stock beat the index? How did the stocks and the
    index perform in 2022?

***
# Exercise 3: How good are inflation expectations?

The files `SCE_10pct/SCE_2013.csv`, ..., `SCE_2025.csv` contain a 10% sample of
the New York Fed's
[Survey of Consumer Expectations](https://www.newyorkfed.org/microeconomics/sce).
Each row is one respondent's answer in one survey month: `userid`, `date`
(survey month), `female` (1 = female, 0 = male) and `infl_1y`, the inflation
rate (in percent) the respondent expects over the **next** 12 months. See the
[data description](../../data/SCE_10pct/README.md).

Workshop 5 compared expectations with inflation in the *past*. Here, you compare
each forecast with the inflation that *actually happened* afterwards.

1.  Load all 13 files in a loop and concatenate them into one DataFrame with a
    clean row index. Report the number of rows, the number of different
    respondents and the first and last survey month.

    *Hint:* [`nunique()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html) counts distinct values.

2.  Many respondents answer in several months. Use `groupby('userid')` with
    `size()` to count how often each respondent appears. How many answered only
    once, and what is the maximum number of answers?

3.  Load `FRED/CPI.csv`, parse `DATE` as dates and rename the column to `date` so
    it matches the survey. Compute the *realized* inflation over the following 12
    months:
    $$
    \text{Realized}_t = \left( \frac{CPI_{t+12}}{CPI_t} - 1 \right) \times 100
    $$
    *Hint:* `shift(-12)` moves values 12 rows *up*, so each row gets the value
    from 12 months later.

4.  Merge the realized inflation onto the survey answers with a left join on
    `date`. How many answers have no realized inflation, and from which survey
    months? Explain why for each month.

    *Hint:* Look at the last date in `CPI.csv` and check whether any CPI values
    are missing.

5.  Drop the answers without realized inflation and compute the forecast error,
    `infl_1y - Realized`. What is the mean forecast error, and what share of
    answers over-predicted inflation?

6.  Compute the mean forecast error by survey year and by gender. In which years
    were people's forecasts closest to the realized inflation? Compare the mean
    and median error for men and women. Why might the two statistics give
    different impressions?